# Custom CNN Fire Detection Model (128x128 Resolution & Data Augmentation)
Enhanced Convolutional Neural Network for detecting fire in images with BatchNormalization, Dropout, Data Augmentation, and Adam optimizer.

In [ ]:
import glob
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import cv2
import matplotlib.pyplot as plt

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)
os.environ["PYTHONHASHSEED"] = "0"

In [ ]:
# Download and unzip dataset
!pip install --upgrade --quiet gdown
!gdown 1YNzR-Pgo654scOwO_cpYmtEkaDXKd96f -O fire_dataset.zip
!unzip -o fire_dataset.zip -d fire_dataset

In [ ]:
# Image Resolution set to 128x128 for rich feature extraction
IMG_SIZE = (128, 128)

def prepare_image(image):
    resized = cv2.resize(image, IMG_SIZE)
    return cv2.cvtColor(resized, cv2.COLOR_BGR2RGB) / 255.0

def get_label(address):
    return os.path.basename(os.path.dirname(address))

X_processed = []
labels = []
dataset_path = "fire_dataset/fire_dataset"

for address in glob.glob(dataset_path + "/**/*"):
    try:
        image_original = cv2.imread(address)
        if image_original is not None:
            image_processed = prepare_image(image_original)
            X_processed.append(image_processed)
            labels.append(get_label(address))
    except Exception as e:
        continue

np_x = np.array(X_processed)
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(labels)
categ_labels = to_categorical(encoded_labels)

print(f"Processed {len(np_x)} images with shape {np_x.shape}")
print("Classes:", label_encoder.classes_)

In [ ]:
# Train / Test Split
x_train, x_test, y_train, y_test = train_test_split(np_x, categ_labels, test_size=0.2, random_state=42)
print(f"Train shape: {x_train.shape}, Test shape: {x_test.shape}")

In [ ]:
# Data Augmentation Layer
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05)
], name="data_augmentation")

# Advanced Custom CNN Model Architecture
model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    data_augmentation,
    
    # Block 1
    layers.Conv2D(32, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(32, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPool2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 2
    layers.Conv2D(64, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Conv2D(64, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPool2D((2, 2)),
    layers.Dropout(0.25),
    
    # Block 3
    layers.Conv2D(128, (3, 3), padding="same"),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.MaxPool2D((2, 2)),
    layers.Dropout(0.3),
    
    # Fully Connected Head
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(2, activation="softmax")
])

# Compile model with Adam optimizer and categorical crossentropy
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [ ]:
# Callbacks for training
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)
]

# Train model with 25 epochs
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=25,
    batch_size=32,
    callbacks=callbacks
)

In [ ]:
# Plot Accuracy and Loss curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy", color="blue")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy", color="orange")
plt.title("Model Accuracy (128x128 CNN)")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss", color="blue")
plt.plot(history.history["val_loss"], label="Validation Loss", color="orange")
plt.title("Model Loss (128x128 CNN)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = model.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_test_classes = np.argmax(y_test, axis=1)
class_names = label_encoder.classes_

print("Classification Report:")
print(classification_report(y_test_classes, y_pred_classes, target_names=class_names))

cm = confusion_matrix(y_test_classes, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix (128x128 Custom CNN)")
plt.show()